In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_0.pickle

trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['responses_df_2022', 'responses_df_2022_cell10']
me:  16
me:  16
trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  21
me:  6
me:  17
me:  10
me:  8
trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  21
me:  6
me:  17
me:  10
me:  8
trying: ['responses_df_2020', 'df']
me:  21
me:  21
trying: ['responses_df_2020', 'df']
me:  21
me:  21
trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  21
me:  6
me:  17
me:  10
me:  8
trying: ['title_for_x_axis_cell33', 'title_for_x_axis', 'title_for_x_axis_cell29', 'title_for_x_axis_cell22', 'title_for_x_axis_cell20']
me:  21
me:  6
me:  17
me:  10
me:  8
trying: ['combine_subset_of_data_from_multiple_y

me:  17
trying: ['responses_df_2017']
me:  17
trying: ['mapping_1_2_to_1_3']
me:  21
trying: ['responses_df_2019_cell10']
me:  21
trying: ['question_of_interest_cell33']
me:  21
trying: ['grab_subset_of_data_27']
me:  15
trying: ['question_of_interest_cell18']
me:  6
trying: ['grab_subset_of_data_34']
me:  22
trying: ['question_of_interest_cell27']
me:  15
trying: ['combine_subset_of_data_from_multiple_years_33']
me:  21
trying: ['country_df_combined']
me:  6
trying: ['online_learning_platforms_df_2022_cell27']
me:  15
trying: ['mapping_2018']
me:  21
trying: ['country_df_combined_cell18']
me:  6
trying: ['responses_df_2018_cell10']
me:  21
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['question_name']
me:  6
trying: ['programming_df_combined']
me:  21
trying: ['count_then_return_percent_18']
me:  6
trying: ['question_name_alternate_cell18']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['question_name_alternate']
me:  6
trying: ['title_

me:  0
trying: ['sns']
me:  0
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['percentages']
me:  5
trying: ['age_df_combined']
me:  8
trying: ['responses_in_order_cell28']
me:  16
trying: ['count_then_return_percent_28']
me:  16
trying: ['responses_df_2018']
me:  1
trying: ['percentages_cell28']
me:  16
trying: ['responses_df_2019']
me:  1
trying: ['question_of_interest_cell28']
me:  16
trying: ['count_then_return_percent_22']
me:  10
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  10
trying: ['percentages_per_country_df']
me:  4
trying: ['add_year_column_to_dataframes_22']
me:  10
trying: ['question_of_interest_cell22']
me:  10
trying: ['age_df_combined_cell22']
me:  10
trying: ['file_name_2017']
me:  1
trying: ['base_dir_2019']
me:  1
trying: ['factor']
me:  1
trying: ['directory']
me:  1
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['base_dir_2020']
me:  1
trying: ['file_name_2019']
me:  1
trying: ['fi

In [3]:
%%RecordEvent
%%time
### cell 23 ###

def grab_subset_of_data_35(original_df, question_of_interest):
    return (
        original_df
            .filter(like=question_of_interest, axis=1)
            .rename(columns=lambda c: c.rsplit('-', 1)[-1].lstrip())
            .dropna(how='all')
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest, include_2017=False):
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs = [
        responses_df_2018_cell10,
        responses_df_2019_cell10,
        responses_df_2020,
        responses_df_2021,
        responses_df_2022_cell10
    ]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)

    # Vectorized concatenation with year keys
    year_df_map = dict(zip(years, dfs))
    combined = (
        pd.concat(year_df_map, axis=0, names=['year'])
          .filter(like=question_of_interest, axis=1)
          .rename(columns=lambda c: c.rsplit('-', 1)[-1].lstrip())
          .dropna(how='all')
          .reset_index(level='year')
          .reset_index(drop=True)
    )
    counts = (
        combined
          .groupby('year')
          .count()
          .reindex(years, fill_value=0)
          .reset_index()
    )
    return combined, counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Compute respondent totals once and broadcast
    totals = df['year'].value_counts().reindex(df_counts['year']).fillna(0)
    return (
        df_counts
          .set_index('year')
          .div(totals, axis=0)
          .mul(100)
          .reset_index()
    )

# Main pipeline
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'
language_df_combined, language_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined,
    language_df_combined_counts
)

# Select & reshape
langs = ['year', 'Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']
language_df_combined_percentages_cell35 = language_df_combined_percentages[langs]

df_cell35 = (
    language_df_combined_percentages_cell35
      .melt(id_vars='year', value_vars=langs[1:])
      .sort_values(['year', 'value'])
      .rename(columns={'variable': ''})
)

df_cell35.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 10 to 4
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    40 non-null     object 
 1           40 non-null     object 
 2   value   40 non-null     float64
dtypes: float64(1), object(2)
memory usage: 1.2+ KB
CPU times: user 422 ms, sys: 63.8 ms, total: 486 ms
Wall time: 486 ms


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_23_try_2.pickle

migration speed (bps): 750868429.3970217


---------------------------
variables to migrate:
USA_responses_df_2022 3060721
question_of_interest_cell24 116
percentages_cell24 151
title_for_y_axis_cell24 65
np 72
learning_platform_df_combined_percentages_cell26 849
warnings 72
question_of_interest_alternate 123
go 72
df_cell26 5761
px 72
sns 72
learning_platform_df_combined_counts 1089
responses_df_2021 34255607
learning_platform_df_combined_percentages 1089
responses_df_2017 15719078
add_year_column_to_dataframes_26 144
grab_subset_of_data_34 144
grab_subset_of_data_26 144
convert_df_of_counts_to_percentages_26 144
question_of_interest_cell26 117
responses_df_2019_cell10 15975586
learning_platform_df_combined 5297884
question_of_interest_cell33 114
combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26 144
title_for_x_axis_cell33 49
responses_df_2020 25517727
combine_subset_of_data_from_multiple_years_33 144
mapping_2018 360
df 25517727
responses_df_2018_cell10 32065472
programming_df_combi

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['responses_df_2019', 'responses_df_2022', 'responses_df_2020', 'responses_df_2021', 'responses_df_2018']
Intermediate variables ['file_name_2022', 'base_dir_2020', 'file_name_2019', 'responses_df_2017', 'directory_cell8', 'base_dir_2022', 'factor', 'file_name_2018', 'base_dir_2021', 'directory', 'base_dir_2019', 'base_dir_2017', 'file_name_2021', 'base_dir_2018', 'file_name_2017', 'file_name_2020']
Future variables ['responses_df_2017', 'responses_df_2022_cell10', 'percentages']
Modified dataframes
======= Cell 2 =======
Input variables ['responses_df_2022', 'responses_df_2019', 'responses_df_2018']
Active variables ['responses_df_2022', 'responses_df_2022_cell10', 'responses_df_2019_cell10']
Intermediate variables ['responses_df_2018_cell10']
Future variables ['percentages', 'responses_df_2017', 'res

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_23_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[23], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_23.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
